# 03 · Labels – ESC-50 + Combined
Map ESC-50 categories to CityPulse labels, combine with US8K labels, and verify.

In [1]:
# interpreter & imports
import sys
print(sys.executable)
import pandas as pd
from pathlib import Path
print('imports OK')

C:\Users\LENOVO\miniconda3\envs\citypulse\python.exe


imports OK


In [2]:
# paths
ROOT   = Path.cwd().parent
ESC50  = ROOT / 'data' / 'raw' / 'ESC-50'
PROC   = ROOT / 'data' / 'processed'
print('ROOT :', ROOT)
print('ESC50 exists:', ESC50.exists())
print('PROC  exists:', PROC.exists())

ROOT : C:\Users\LENOVO\Desktop\CityPulse
ESC50 exists: True
PROC  exists: True


In [3]:
# Cell 3 – Part A step 1: load esc50.csv, print sorted categories, confirm spellings
esc_meta = pd.read_csv(ESC50 / 'meta' / 'esc50.csv')
print(f'esc50.csv shape: {esc_meta.shape}')
print('Columns:', esc_meta.columns.tolist())

all_cats = sorted(esc_meta['category'].unique().tolist())
print(f'\nAll {len(all_cats)} unique categories (sorted):')
for c in all_cats:
    print(f'  {c}')

# confirm exact spellings required by the mapping
REQUIRED = ['car_horn', 'engine', 'chainsaw', 'hand_saw', 'siren']
print('\nSpelling check:')
all_ok = True
for name in REQUIRED:
    found = name in all_cats
    print(f'  {name!r}: {"OK" if found else "*** NOT FOUND – STOP ***"}')
    if not found:
        all_ok = False

if not all_ok:
    raise ValueError('One or more required category names are not spelled correctly in esc50.csv – see above.')
print('All required categories confirmed.')

esc50.csv shape: (2000, 7)
Columns: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take']

All 50 unique categories (sorted):
  airplane
  breathing
  brushing_teeth
  can_opening
  car_horn
  cat
  chainsaw
  chirping_birds
  church_bells
  clapping
  clock_alarm
  clock_tick
  coughing
  cow
  crackling_fire
  crickets
  crow
  crying_baby
  dog
  door_wood_creaks
  door_wood_knock
  drinking_sipping
  engine
  fireworks
  footsteps
  frog
  glass_breaking
  hand_saw
  helicopter
  hen
  insects
  keyboard_typing
  laughing
  mouse_click
  pig
  pouring_water
  rain
  rooster
  sea_waves
  sheep
  siren
  sneezing
  snoring
  thunderstorm
  toilet_flush
  train
  vacuum_cleaner
  washing_machine
  water_drops
  wind

Spelling check:
  'car_horn': OK
  'engine': OK
  'chainsaw': OK
  'hand_saw': OK
  'siren': OK
All required categories confirmed.


In [4]:
# Cell 4 – Part A steps 2-4: map, build DataFrame, save labels_esc50.csv
ESC50_MAP = {
    'car_horn': 'traffic_horn',
    'engine':   'traffic_horn',
    'chainsaw': 'construction',
    'hand_saw': 'construction',
    'siren':    'siren',
}

kept = esc_meta[esc_meta['category'].isin(ESC50_MAP)].copy()
dropped_n = len(esc_meta) - len(kept)
print(f'ESC-50 rows kept: {len(kept)}, dropped: {dropped_n}, total: {len(esc_meta)}')

df_esc = pd.DataFrame({
    'path':           'data/raw/ESC-50/audio/' + kept['filename'],
    'source':         'esc50',
    'fold':           kept['fold'].values,
    'original_class': kept['category'].values,
    'app_class':      kept['category'].map(ESC50_MAP).values,
})
df_esc = df_esc.reset_index(drop=True)

print('df_esc shape:', df_esc.shape)
print('Columns:', df_esc.columns.tolist())
print(df_esc.head(3).to_string())

out_esc = PROC / 'labels_esc50.csv'
df_esc.to_csv(out_esc, index=False)
print(f'\nSaved: {out_esc}')

ESC-50 rows kept: 200, dropped: 1800, total: 2000
df_esc shape: (200, 5)
Columns: ['path', 'source', 'fold', 'original_class', 'app_class']
                                      path source  fold original_class     app_class
0  data/raw/ESC-50/audio/1-116765-A-41.wav  esc50     1       chainsaw  construction
1   data/raw/ESC-50/audio/1-17124-A-43.wav  esc50     1       car_horn  traffic_horn
2   data/raw/ESC-50/audio/1-18527-A-44.wav  esc50     1         engine  traffic_horn

Saved: C:\Users\LENOVO\Desktop\CityPulse\data\processed\labels_esc50.csv


In [5]:
# Cell 5 – Part B: load both CSVs from disk, concatenate, save labels.csv
df_us8k_disk = pd.read_csv(PROC / 'labels_us8k.csv')
df_esc_disk  = pd.read_csv(PROC / 'labels_esc50.csv')

print('labels_us8k.csv shape:', df_us8k_disk.shape)
print('labels_esc50.csv shape:', df_esc_disk.shape)

df_combined = pd.concat([df_us8k_disk, df_esc_disk], ignore_index=True)
print('combined shape:', df_combined.shape)

out_combined = PROC / 'labels.csv'
df_combined.to_csv(out_combined, index=False)
print(f'Saved: {out_combined}')

labels_us8k.csv shape: (6358, 5)
labels_esc50.csv shape: (200, 5)
combined shape: (6558, 5)
Saved: C:\Users\LENOVO\Desktop\CityPulse\data\processed\labels.csv


In [6]:
# Cell 6 – Part C: ALL verifications on fresh disk reloads
# -----------------------------------------------------------------
# Fresh reload – never reuse in-memory DataFrames
# -----------------------------------------------------------------
lesc  = pd.read_csv(PROC / 'labels_esc50.csv')
lcomb = pd.read_csv(PROC / 'labels.csv')

EXPECTED_COLS = ['path', 'source', 'fold', 'original_class', 'app_class']
EXPECTED_APPS = {'traffic_horn', 'construction', 'siren', 'music_loudspeaker', 'hvac'}

# ---- labels_esc50 assertions ----------------------------------------
assert lesc.shape == (200, 5), \
    f'labels_esc50 shape: expected (200,5), got {lesc.shape}'

cat_counts = lesc['app_class'].value_counts()
assert cat_counts.get('traffic_horn', 0) == 80, \
    f'ESC-50 traffic_horn: expected 80, got {cat_counts.get("traffic_horn",0)}'
assert cat_counts.get('construction', 0) == 80, \
    f'ESC-50 construction: expected 80, got {cat_counts.get("construction",0)}'
assert cat_counts.get('siren', 0) == 40, \
    f'ESC-50 siren: expected 40, got {cat_counts.get("siren",0)}'

orig_counts = lesc['original_class'].value_counts()
for cat in ['car_horn', 'engine', 'chainsaw', 'hand_saw', 'siren']:
    assert orig_counts.get(cat, 0) == 40, \
        f'ESC-50 original_class {cat!r}: expected 40, got {orig_counts.get(cat,0)}'

fold_counts_esc = lesc['fold'].value_counts().sort_index()
assert set(fold_counts_esc.index.tolist()) == {1,2,3,4,5}, \
    f'ESC-50 folds: expected {{1..5}}, got {set(fold_counts_esc.index)}'
for f in range(1, 6):
    assert fold_counts_esc[f] == 40, \
        f'ESC-50 fold {f}: expected 40, got {fold_counts_esc[f]}'

# ---- labels.csv assertions ------------------------------------------
assert lcomb.shape == (6558, 5), \
    f'labels.csv shape: expected (6558,5), got {lcomb.shape}'

assert lcomb.columns.tolist() == EXPECTED_COLS, \
    f'labels.csv columns: expected {EXPECTED_COLS}, got {lcomb.columns.tolist()}'

src_counts = lcomb['source'].value_counts()
assert src_counts.get('us8k', 0) == 6358, \
    f'source us8k: expected 6358, got {src_counts.get("us8k",0)}'
assert src_counts.get('esc50', 0) == 200, \
    f'source esc50: expected 200, got {src_counts.get("esc50",0)}'

# fold dtype is integer
assert pd.api.types.is_integer_dtype(lcomb['fold']), \
    f'fold dtype is not integer: {lcomb["fold"].dtype}'

# fold ranges by source
us8k_folds  = set(lcomb.loc[lcomb['source']=='us8k',  'fold'].unique())
esc50_folds = set(lcomb.loc[lcomb['source']=='esc50', 'fold'].unique())
assert us8k_folds  == set(range(1,11)), \
    f'US8K fold range: expected 1..10, got {sorted(us8k_folds)}'
assert esc50_folds == set(range(1,6)), \
    f'ESC-50 fold range: expected 1..5, got {sorted(esc50_folds)}'

# 0 NaN anywhere
nan_total = lcomb.isna().sum().sum()
assert nan_total == 0, \
    f'NaN count: expected 0, got {nan_total}\n{lcomb.isna().sum()}'

# 0 duplicate paths
n_dup = lcomb['path'].duplicated().sum()
assert n_dup == 0, \
    f'Duplicate paths: expected 0, got {n_dup}'

# every path exists on disk
missing = lcomb['path'].apply(lambda p: not (ROOT / p).exists())
assert missing.sum() == 0, \
    f'Missing files: expected 0, got {missing.sum()}\n{lcomb.loc[missing, "path"].head(5)}'

# no path contains backslash, 'C:', or 'Users'
bad_paths = lcomb['path'].str.contains(r'\\|C:|Users', regex=True)
assert not bad_paths.any(), \
    f'Paths with backslash/C:/Users: {lcomb.loc[bad_paths, "path"].head(5).tolist()}'

# app_class set
got_apps = set(lcomb['app_class'].unique())
assert got_apps == EXPECTED_APPS, \
    f'app_class set: expected {EXPECTED_APPS}, got {got_apps}'

# crosstab assertions  (app_class x source)
ct = pd.crosstab(lcomb['app_class'], lcomb['source'])
EXPECTED_CT = {
    ('construction',      'us8k'): 2000, ('construction',      'esc50'): 80,
    ('hvac',              'us8k'): 1000, ('hvac',              'esc50'): 0,
    ('music_loudspeaker', 'us8k'): 1000, ('music_loudspeaker', 'esc50'): 0,
    ('siren',             'us8k'):  929, ('siren',             'esc50'): 40,
    ('traffic_horn',      'us8k'): 1429, ('traffic_horn',      'esc50'): 80,
}
for (app, src), exp_val in EXPECTED_CT.items():
    got_val = int(ct.loc[app, src]) if (app in ct.index and src in ct.columns) else 0
    assert got_val == exp_val, \
        f'crosstab[{app!r}][{src!r}]: expected {exp_val}, got {got_val}'

# no path appears with both sources
us8k_paths  = set(lcomb.loc[lcomb['source']=='us8k',  'path'])
esc50_paths = set(lcomb.loc[lcomb['source']=='esc50', 'path'])
overlap = us8k_paths & esc50_paths
assert len(overlap) == 0, \
    f'Paths appearing in both sources: {list(overlap)[:5]}'

print('=' * 55)
print('ALL ASSERTIONS PASSED')
print('=' * 55)

print('\n=== Crosstab: app_class x source ===')
print(ct.to_string())

print(f'\n=== labels.csv shape: {lcomb.shape} ===')

print('\n=== head(3) ===')
print(lcomb.head(3).to_string())

print('\n=== tail(3) ===')
print(lcomb.tail(3).to_string())

ALL ASSERTIONS PASSED

=== Crosstab: app_class x source ===
source             esc50  us8k
app_class                     
construction          80  2000
hvac                   0  1000
music_loudspeaker      0  1000
siren                 40   929
traffic_horn          80  1429

=== labels.csv shape: (6558, 5) ===

=== head(3) ===
                                                  path source  fold original_class     app_class
0  data/raw/UrbanSound8K/audio/fold10/100648-1-0-0.wav   us8k    10       car_horn  traffic_horn
1  data/raw/UrbanSound8K/audio/fold10/100648-1-1-0.wav   us8k    10       car_horn  traffic_horn
2  data/raw/UrbanSound8K/audio/fold10/100648-1-2-0.wav   us8k    10       car_horn  traffic_horn

=== tail(3) ===
                                         path source  fold original_class     app_class
6555  data/raw/ESC-50/audio/5-253101-A-49.wav  esc50     5       hand_saw  construction
6556  data/raw/ESC-50/audio/5-253101-B-49.wav  esc50     5       hand_saw  construction
